# Phase 3 — Data Cleaning & Transformation

## 3.1 Initialise Cleaning Environment

The source datasets are loaded from the raw data directory to begin controlled
data cleaning and transformation.

Raw source files are treated as immutable. Cleaning operations will be performed
on DataFrames in memory, with processed outputs written separately to the
designated output directory.

In [1]:
from pathlib import Path
import pandas as pd

# Define project data paths
raw_data_path = Path("../data/raw")
output_data_path = Path("../data/output")

# Load all raw CSV datasets
source_files = list(raw_data_path.glob("*.csv"))

datasets = {}

for file in source_files:
    dataset_name = file.stem
    datasets[dataset_name] = pd.read_csv(file)

print(f"Successfully loaded {len(datasets)} datasets for cleaning.")

Successfully loaded 11 datasets for cleaning.


## 3.2 Create Working Copies

Working copies of the source datasets are created before applying transformations.
This preserves the original loaded data in memory and enables comparison between
the source and cleaned versions during validation.

In [2]:
# Create working copies for cleaning and transformation

cleaned_datasets = {
    name: df.copy()
    for name, df in datasets.items()
}

print(f"Created {len(cleaned_datasets)} working datasets.")

Created 11 working datasets.


## 3.3 Data Cleaning Strategy

The following cleaning and transformation rules are based on the findings from
the source data profiling phase:

1. Remove exact duplicate records from the geolocation dataset while retaining
   legitimate records where the same ZIP-code prefix has different coordinates.

2. Convert identified date and timestamp attributes to appropriate datetime
   data types.

3. Preserve missing optional review titles and comments, as these may represent
   valid customer behaviour rather than data quality errors.

4. Preserve missing delivery dates where the order status indicates that
   delivery has not occurred.

5. Flag delivered orders with missing customer delivery dates as data quality
   exceptions rather than imputing artificial dates.

6. Investigate and flag zero-value payments rather than automatically removing
   them.

7. Investigate and flag zero or negative product weights rather than replacing
   values without supporting evidence.

8. Preserve highly incomplete closed-deal attributes unless there is a justified
   analytical or technical reason to remove them.

9. Validate each transformation by comparing relevant metrics before and after
   cleaning.

10. Write cleaned datasets to the output directory without modifying the raw
    source files.

## 3.4 Remove Exact Geolocation Duplicates

Exact duplicate records identified in the geolocation dataset are removed.

Deduplication is performed across the complete record rather than using ZIP-code
prefix alone, ensuring that legitimate geographic records with the same ZIP-code
prefix but different coordinates are retained.

In [3]:
# Remove exact duplicate geolocation records

geo_df = cleaned_datasets["geolocation_dataset"]

rows_before = len(geo_df)

geo_df = geo_df.drop_duplicates().copy()

rows_after = len(geo_df)
rows_removed = rows_before - rows_after

# Update the cleaned dataset
cleaned_datasets["geolocation_dataset"] = geo_df

print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after cleaning  : {rows_after:,}")
print(f"Duplicate rows removed: {rows_removed:,}")

Rows before cleaning : 1,000,163
Rows after cleaning  : 738,332
Duplicate rows removed: 261,831


## 3.5 Validate Geolocation Deduplication

The geolocation transformation is validated to confirm that all exact duplicate
records were removed while the original raw dataset remained unchanged.

In [6]:
# Validate geolocation deduplication

original_geo = datasets["geolocation_dataset"]
cleaned_geo = cleaned_datasets["geolocation_dataset"]

validation = {
    "Original row count": len(original_geo),
    "Cleaned row count": len(cleaned_geo),
    "Rows removed": len(original_geo) - len(cleaned_geo),
    "Original duplicate rows": original_geo.duplicated().sum(),
    "Remaining duplicate rows": cleaned_geo.duplicated().sum()
}

for check, result in validation.items():
    print(f"{check:<30}: {result:,}")

Original row count            : 1,000,163
Cleaned row count             : 738,332
Rows removed                  : 261,831
Original duplicate rows       : 261,831
Remaining duplicate rows      : 0


## 3.6 Convert Date and Timestamp Columns

Date and timestamp attributes are converted from text-based representations to
Pandas datetime types.

Conversion is applied to the cleaned working datasets to support reliable date
filtering, comparison, aggregation, and downstream analysis.

In [7]:
# Convert date and timestamp columns to datetime

date_conversion_summary = []

for name, df in cleaned_datasets.items():

    date_columns = [
        col for col in df.columns
        if "date" in col.lower() or "timestamp" in col.lower()
    ]

    for col in date_columns:

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        date_conversion_summary.append({
            "dataset": name,
            "column": col,
            "dtype_after": str(df[col].dtype),
            "null_values": df[col].isna().sum()
        })

date_conversion_df = pd.DataFrame(date_conversion_summary)

display(date_conversion_df)

,dataset,column,dtype_after,null_values
0,closed_deals_dataset,won_date,datetime64[us],0
1,marketing_qualified_leads_dataset,first_contact_date,datetime64[us],0
2,orders_dataset,order_purchase_timestamp,datetime64[us],0
3,orders_dataset,order_delivered_carrier_date,datetime64[us],1783
4,orders_dataset,order_delivered_customer_date,datetime64[us],2965
5,orders_dataset,order_estimated_delivery_date,datetime64[us],0
6,order_items_dataset,shipping_limit_date,datetime64[us],0
7,order_reviews_dataset,review_creation_date,datetime64[us],0
8,order_reviews_dataset,review_answer_timestamp,datetime64[us],0


## 3.7 Convert Additional Order Date Field

The `order_approved_at` attribute was not captured by the automated date-column
detection rule because its column name does not contain "date" or "timestamp".

The field is explicitly converted to datetime to ensure consistent handling of
all order lifecycle timestamps.

In [8]:
# Convert order_approved_at explicitly

orders_clean = cleaned_datasets["orders_dataset"]

orders_clean["order_approved_at"] = pd.to_datetime(
    orders_clean["order_approved_at"],
    errors="coerce"
)

print(f"Data type: {orders_clean['order_approved_at'].dtype}")
print(f"Null values: {orders_clean['order_approved_at'].isna().sum():,}")

Data type: datetime64[us]
Null values: 160


## 3.8 Add Data Quality Flags

Data quality flags are added for identified business-rule exceptions.

Rather than deleting or imputing values without supporting evidence, the original
values are retained and affected records are explicitly flagged for downstream
validation and analysis.

In [9]:
# Add data quality flags for identified exceptions

orders_clean = cleaned_datasets["orders_dataset"]
payments_clean = cleaned_datasets["order_payments_dataset"]
products_clean = cleaned_datasets["products_dataset"]

# Flag delivered orders with missing customer delivery dates
orders_clean["dq_missing_delivery_date"] = (
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].isna())
)

# Flag zero-value payments
payments_clean["dq_zero_payment"] = (
    payments_clean["payment_value"] == 0
)

# Flag zero or negative product weights
products_clean["dq_invalid_weight"] = (
    products_clean["product_weight_g"] <= 0
)

print(
    "Delivered orders missing delivery date:",
    orders_clean["dq_missing_delivery_date"].sum()
)

print(
    "Zero-value payments:",
    payments_clean["dq_zero_payment"].sum()
)

print(
    "Zero/non-positive product weights:",
    products_clean["dq_invalid_weight"].sum()
)

Delivered orders missing delivery date: 8
Zero-value payments: 9
Zero/non-positive product weights: 4


## 3.9 Validate Data Quality Flags

The newly created data quality flags are validated against the exception counts
identified during source profiling to confirm that the transformation logic
correctly identifies the expected records.

In [12]:
# Validate data quality flags

flag_validation = pd.DataFrame({
    "quality_check": [
        "Delivered order missing delivery date",
        "Zero-value payment",
        "Zero/non-positive product weight"
    ],
    "expected_count": [
        8,
        9,
        4
    ],
    "actual_count": [
        orders_clean["dq_missing_delivery_date"].sum(),
        payments_clean["dq_zero_payment"].sum(),
        products_clean["dq_invalid_weight"].sum()
    ]
})

flag_validation["passed"] = (
    flag_validation["expected_count"] ==
    flag_validation["actual_count"]
)

display(flag_validation)

,quality_check,expected_count,actual_count,passed
0,Delivered order missing delivery date,8,8,True
1,Zero-value payment,9,9,True
2,Zero/non-positive product weight,4,4,True


## 3.10 Missing Value Treatment

Missing values are treated according to their business meaning rather than using
a blanket deletion or imputation strategy.

Optional review text fields are retained as null where no customer comment was
provided. Missing order lifecycle dates are preserved where consistent with the
order status. Highly incomplete sales attributes and missing product attributes
are also retained where no reliable replacement value can be derived.

This approach avoids introducing artificial information into the cleaned data.

## 3.11 Standardise Text Attributes

Text attributes are standardised by removing leading and trailing whitespace.
Null values are preserved and text content is otherwise left unchanged to avoid
altering legitimate source information.

In [14]:
# Remove leading and trailing whitespace from genuine text columns

text_cleaning_summary = []

for name, df in cleaned_datasets.items():

    for col in df.columns:

        # Only process columns containing text values
        if pd.api.types.is_string_dtype(df[col]):

            before = df[col].copy()

            df[col] = df[col].apply(
                lambda x: x.strip() if isinstance(x, str) else x
            )

            changed_count = (
                before.fillna("") != df[col].fillna("")
            ).sum()

            if changed_count > 0:
                text_cleaning_summary.append({
                    "dataset": name,
                    "column": col,
                    "values_changed": changed_count
                })

text_cleaning_df = pd.DataFrame(text_cleaning_summary)

if text_cleaning_df.empty:
    print("No leading or trailing whitespace issues detected.")
else:
    display(text_cleaning_df)

,dataset,column,values_changed
0,geolocation_dataset,geolocation_city,1
1,order_reviews_dataset,review_comment_title,1998
2,order_reviews_dataset,review_comment_message,9451


## 3.12 Post-Cleaning Validation

The cleaned datasets are validated against the original source datasets to
confirm that transformations were controlled and that unexpected record loss
has not occurred.

Row counts, duplicate counts, and column counts are compared between the source
and cleaned datasets.

In [15]:
# Compare original and cleaned datasets

validation_summary = []

for name in datasets.keys():

    original_df = datasets[name]
    cleaned_df = cleaned_datasets[name]

    validation_summary.append({
        "dataset": name,
        "original_rows": len(original_df),
        "cleaned_rows": len(cleaned_df),
        "rows_difference": len(cleaned_df) - len(original_df),
        "original_columns": len(original_df.columns),
        "cleaned_columns": len(cleaned_df.columns),
        "remaining_duplicates": cleaned_df.duplicated().sum()
    })

validation_summary_df = pd.DataFrame(validation_summary)

display(validation_summary_df)

,dataset,original_rows,cleaned_rows,rows_difference,original_columns,cleaned_columns,remaining_duplicates
0,closed_deals_dataset,842,842,0,14,14,0
1,customers_dataset,99441,99441,0,5,5,0
2,geolocation_dataset,1000163,738332,-261831,5,5,0
3,marketing_qualified_leads_dataset,8000,8000,0,4,4,0
4,orders_dataset,99441,99441,0,8,9,0
5,order_items_dataset,112650,112650,0,7,7,0
6,order_payments_dataset,103886,103886,0,5,6,0
7,order_reviews_dataset,99224,99224,0,7,7,0
8,products_dataset,32951,32951,0,9,10,0
9,product_category_name_translation,71,71,0,2,2,0


## 3.13 Final Data Quality Validation

Final validation checks are performed before cleaned datasets are written to
the output layer.

The checks confirm that exact duplicates have been removed, expected data
quality flags remain consistent, and cleaned dataset row counts match the
intended transformation rules.

In [16]:
# Final data quality validation

final_checks = {
    "All cleaned datasets have no exact duplicates":
        all(
            df.duplicated().sum() == 0
            for df in cleaned_datasets.values()
        ),

    "Geolocation row reduction is correct":
        (
            len(datasets["geolocation_dataset"])
            - len(cleaned_datasets["geolocation_dataset"])
        ) == 261831,

    "Delivered-order exception count is correct":
        cleaned_datasets["orders_dataset"][
            "dq_missing_delivery_date"
        ].sum() == 8,

    "Zero-payment exception count is correct":
        cleaned_datasets["order_payments_dataset"][
            "dq_zero_payment"
        ].sum() == 9,

    "Invalid-weight exception count is correct":
        cleaned_datasets["products_dataset"][
            "dq_invalid_weight"
        ].sum() == 4
}

final_validation_df = pd.DataFrame(
    final_checks.items(),
    columns=["validation_check", "passed"]
)

display(final_validation_df)

,validation_check,passed
0,All cleaned datasets have no exact duplicates,True
1,Geolocation row reduction is correct,True
2,Delivered-order exception count is correct,True
3,Zero-payment exception count is correct,True
4,Invalid-weight exception count is correct,True


## 3.14 Export Cleaned Datasets

Following successful validation, the cleaned datasets are written to the
designated output directory.

The original raw source files remain unchanged. Cleaned outputs are stored
separately to maintain clear separation between source and processed data.

In [17]:
# Export cleaned datasets to the output directory

output_data_path.mkdir(parents=True, exist_ok=True)

export_summary = []

for name, df in cleaned_datasets.items():

    output_file = output_data_path / f"{name}_cleaned.csv"

    df.to_csv(
        output_file,
        index=False
    )

    export_summary.append({
        "dataset": name,
        "rows_exported": len(df),
        "columns_exported": len(df.columns),
        "output_file": output_file.name
    })

export_summary_df = pd.DataFrame(export_summary)

display(export_summary_df)

,dataset,rows_exported,columns_exported,output_file
0,closed_deals_dataset,842,14,closed_deals_dataset_cleaned.csv
1,customers_dataset,99441,5,customers_dataset_cleaned.csv
2,geolocation_dataset,738332,5,geolocation_dataset_cleaned.csv
3,marketing_qualified_leads_dataset,8000,4,marketing_qualified_leads_dataset_cleaned.csv
4,orders_dataset,99441,9,orders_dataset_cleaned.csv
5,order_items_dataset,112650,7,order_items_dataset_cleaned.csv
6,order_payments_dataset,103886,6,order_payments_dataset_cleaned.csv
7,order_reviews_dataset,99224,7,order_reviews_dataset_cleaned.csv
8,products_dataset,32951,10,products_dataset_cleaned.csv
9,product_category_name_translation,71,2,product_category_name_translation_cleaned.csv


## 3.15 Verify Exported Files

The output directory is checked to confirm that all cleaned datasets were
successfully written following validation.

In [18]:
# Verify exported cleaned files

exported_files = list(output_data_path.glob("*_cleaned.csv"))

print(f"Expected files : {len(cleaned_datasets)}")
print(f"Exported files : {len(exported_files)}")

for file in sorted(exported_files):
    print(file.name)

Expected files : 11
Exported files : 11
closed_deals_dataset_cleaned.csv
customers_dataset_cleaned.csv
geolocation_dataset_cleaned.csv
marketing_qualified_leads_dataset_cleaned.csv
order_items_dataset_cleaned.csv
order_payments_dataset_cleaned.csv
order_reviews_dataset_cleaned.csv
orders_dataset_cleaned.csv
product_category_name_translation_cleaned.csv
products_dataset_cleaned.csv
sellers_dataset_cleaned.csv


## 3.16 Phase 3 Summary

The data cleaning and transformation phase applied controlled, evidence-based
changes to the source datasets while preserving the original raw files.

### Transformations Completed

- Created independent working copies of all 11 source datasets.
- Removed 261,831 exact duplicate records from the geolocation dataset,
  reducing it from 1,000,163 to 738,332 records.
- Preserved legitimate geolocation records where ZIP-code prefixes were repeated
  with different coordinates.
- Converted identified date and timestamp attributes to appropriate datetime
  data types.
- Explicitly converted `order_approved_at`, which was not captured by the
  initial automated date-column detection rule.
- Preserved legitimate missing values where no reliable replacement could be
  derived.
- Added data quality flags for:
  - 8 delivered orders with missing customer delivery dates.
  - 9 zero-value payment records.
  - 4 zero or non-positive product-weight records.
- Removed leading and trailing whitespace from affected text attributes.
- Validated row counts, duplicate removal, data quality flags, and transformation
  results.
- All final automated data quality checks passed.
- Exported and verified 11 cleaned datasets in the processed output directory.

### Outcome

The cleaned datasets provide a validated and traceable processed data layer for
subsequent integration and analytical transformation while maintaining the
original raw source data unchanged.